# Neural Networks with L1 and L2 Regularization

This notebook compares four models on a 10-class synthetic dataset:

| Model | Regularization |
|-------|----------------|
| Model 1 | None (Baseline — overfitting case) |
| Model 2 | L1 (Lasso) |
| Model 3 | L2 (Ridge) |
| Model 4 | L1 + L2 (Elastic Net) |

---

## 1 — Imports

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers
import numpy as np
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

print(f"TensorFlow: {tf.__version__}")

## 2 — Create Dataset

Synthetic 10-class classification with 4 features (all informative, no redundant features).

In [ ]:
np.random.seed(42)
tf.random.set_seed(42)

X, y = make_classification(
    n_samples=1000,
    n_features=4,
    n_informative=4,
    n_redundant=0,
    n_classes=10,
    n_clusters_per_class=1,
    random_state=42
)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"X_train shape : {X_train.shape}")
print(f"X_test  shape : {X_test.shape}")
print(f"Classes       : {np.unique(y)}")

## 3 — Model 1: No Regularization (Baseline)

This model has no regularization. It is expected to overfit — training accuracy
will be much higher than validation accuracy.

In [ ]:
print("=" * 60)
print("MODEL 1: NO REGULARIZATION (BASELINE)")
print("=" * 60)

model_overfit = keras.Sequential([
    keras.Input(shape=(4,)),
    layers.Dense(3, activation='relu'),
    layers.Dense(4, activation='relu'),
    layers.Dense(10, activation='softmax')
], name="Baseline")

model_overfit.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

history_overfit = model_overfit.fit(
    X_train, y_train,
    epochs=200,
    batch_size=32,
    validation_data=(X_test, y_test),
    verbose=0
)

train_loss_overfit = history_overfit.history['loss'][-1]
train_acc_overfit  = history_overfit.history['accuracy'][-1]
val_loss_overfit   = history_overfit.history['val_loss'][-1]
val_acc_overfit    = history_overfit.history['val_accuracy'][-1]

print(f"\nFinal Training Accuracy   : {train_acc_overfit:.4f}")
print(f"Final Training Loss       : {train_loss_overfit:.4f}")
print(f"Final Validation Accuracy : {val_acc_overfit:.4f}")
print(f"Final Validation Loss     : {val_loss_overfit:.4f}")
print(f"Overfitting Gap (Acc)     : {train_acc_overfit - val_acc_overfit:.4f}")
print(f"Overfitting Gap (Loss)    : {val_loss_overfit - train_loss_overfit:.4f}")

### Baseline — Loss Curve

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

ax.plot(history_overfit.history['loss'],     label='Train Loss', linewidth=2)
ax.plot(history_overfit.history['val_loss'], label='Val Loss',   linewidth=2)
ax.set_title('No Regularization — Loss Curve', fontsize=14, fontweight='bold')
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 4 — Model 2: L1 Regularization

L1 adds `λ × Σ|w|` to the loss. The gradient is `λ × sign(w)` — a constant force
toward zero regardless of weight magnitude. This can push weights to exactly 0 (sparsity).

In [ ]:
print("=" * 60)
print("MODEL 2: L1 REGULARIZATION")
print("=" * 60)

model_l1 = keras.Sequential([
    keras.Input(shape=(4,)),
    layers.Dense(3, activation='relu', kernel_regularizer=regularizers.L1(0.01)),
    layers.Dense(4, activation='relu', kernel_regularizer=regularizers.L1(0.01)),
    layers.Dense(10, activation='softmax')
], name="L1_Regularization")

model_l1.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

history_l1 = model_l1.fit(
    X_train, y_train,
    epochs=200,
    batch_size=32,
    validation_data=(X_test, y_test),
    verbose=0
)

train_loss_l1 = history_l1.history['loss'][-1]
train_acc_l1  = history_l1.history['accuracy'][-1]
val_loss_l1   = history_l1.history['val_loss'][-1]
val_acc_l1    = history_l1.history['val_accuracy'][-1]

print(f"\nFinal Training Accuracy   : {train_acc_l1:.4f}")
print(f"Final Training Loss       : {train_loss_l1:.4f}")
print(f"Final Validation Accuracy : {val_acc_l1:.4f}")
print(f"Final Validation Loss     : {val_loss_l1:.4f}")
print(f"Overfitting Gap (Acc)     : {train_acc_l1 - val_acc_l1:.4f}")
print(f"Overfitting Gap (Loss)    : {val_loss_l1 - train_loss_l1:.4f}")

### L1 — Loss Curve

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

ax.plot(history_l1.history['loss'],     label='Train Loss', linewidth=2)
ax.plot(history_l1.history['val_loss'], label='Val Loss',   linewidth=2)
ax.set_title('L1 Regularization — Loss Curve', fontsize=14, fontweight='bold')
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 5 — Model 3: L2 Regularization

L2 adds `λ × Σw²` to the loss. The gradient is `2λw` — proportional to weight size.
Large weights get pulled down hard; small weights get a gentle nudge. Weights shrink
but rarely reach exactly zero (weight decay).

In [ ]:
print("=" * 60)
print("MODEL 3: L2 REGULARIZATION")
print("=" * 60)

model_l2 = keras.Sequential([
    keras.Input(shape=(4,)),
    layers.Dense(3, activation='relu', kernel_regularizer=regularizers.L2(0.01)),
    layers.Dense(4, activation='relu', kernel_regularizer=regularizers.L2(0.01)),
    layers.Dense(10, activation='softmax')
], name="L2_Regularization")

model_l2.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

history_l2 = model_l2.fit(
    X_train, y_train,
    epochs=200,
    batch_size=32,
    validation_data=(X_test, y_test),
    verbose=0
)

train_loss_l2 = history_l2.history['loss'][-1]
train_acc_l2  = history_l2.history['accuracy'][-1]
val_loss_l2   = history_l2.history['val_loss'][-1]
val_acc_l2    = history_l2.history['val_accuracy'][-1]

print(f"\nFinal Training Accuracy   : {train_acc_l2:.4f}")
print(f"Final Training Loss       : {train_loss_l2:.4f}")
print(f"Final Validation Accuracy : {val_acc_l2:.4f}")
print(f"Final Validation Loss     : {val_loss_l2:.4f}")
print(f"Overfitting Gap (Acc)     : {train_acc_l2 - val_acc_l2:.4f}")
print(f"Overfitting Gap (Loss)    : {val_loss_l2 - train_loss_l2:.4f}")

### L2 — Loss Curve

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

ax.plot(history_l2.history['loss'],     label='Train Loss', linewidth=2)
ax.plot(history_l2.history['val_loss'], label='Val Loss',   linewidth=2)
ax.set_title('L2 Regularization — Loss Curve', fontsize=14, fontweight='bold')
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 6 — Model 4: L1 + L2 Regularization (Elastic Net)

Elastic Net combines both penalties:
```
L_total = CrossEntropy + λ₁·Σ|w| + λ₂·Σw²
```
This gives both sparsity (from L1) and weight shrinkage (from L2).

In [ ]:
print("=" * 60)
print("MODEL 4: L1 + L2 REGULARIZATION (ELASTIC NET)")
print("=" * 60)

model_l1_l2 = keras.Sequential([
    keras.Input(shape=(4,)),
    layers.Dense(3, activation='relu',
                 kernel_regularizer=regularizers.L1L2(l1=0.01, l2=0.01)),
    layers.Dense(4, activation='relu',
                 kernel_regularizer=regularizers.L1L2(l1=0.01, l2=0.01)),
    layers.Dense(10, activation='softmax')
], name="L1_L2_ElasticNet")

model_l1_l2.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

history_l1_l2 = model_l1_l2.fit(
    X_train, y_train,
    epochs=200,
    batch_size=32,
    validation_data=(X_test, y_test),
    verbose=0
)

train_loss_l1_l2 = history_l1_l2.history['loss'][-1]
train_acc_l1_l2  = history_l1_l2.history['accuracy'][-1]
val_loss_l1_l2   = history_l1_l2.history['val_loss'][-1]
val_acc_l1_l2    = history_l1_l2.history['val_accuracy'][-1]

print(f"\nFinal Training Accuracy   : {train_acc_l1_l2:.4f}")
print(f"Final Training Loss       : {train_loss_l1_l2:.4f}")
print(f"Final Validation Accuracy : {val_acc_l1_l2:.4f}")
print(f"Final Validation Loss     : {val_loss_l1_l2:.4f}")
print(f"Overfitting Gap (Acc)     : {train_acc_l1_l2 - val_acc_l1_l2:.4f}")
print(f"Overfitting Gap (Loss)    : {val_loss_l1_l2 - train_loss_l1_l2:.4f}")

### L1+L2 — Loss Curve

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

ax.plot(history_l1_l2.history['loss'],     label='Train Loss', linewidth=2)
ax.plot(history_l1_l2.history['val_loss'], label='Val Loss',   linewidth=2)
ax.set_title('L1 + L2 Regularization (Elastic Net) — Loss Curve',
             fontsize=14, fontweight='bold')
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 7 — Comparison Summary Table

A smaller **gap** between train and val accuracy means less overfitting.

In [ ]:
print("\n" + "=" * 72)
print("COMPARISON SUMMARY")
print("=" * 72)
print(f"{'Model':<22} {'Train Acc':>10} {'Val Acc':>10} {'Gap (Acc)':>12} {'Gap (Loss)':>12}")
print("-" * 72)
print(f"{'No Regularization':<22} {train_acc_overfit:>10.4f} {val_acc_overfit:>10.4f} "
      f"{train_acc_overfit - val_acc_overfit:>12.4f} "
      f"{val_loss_overfit  - train_loss_overfit:>12.4f}")
print(f"{'L1 Regularization':<22} {train_acc_l1:>10.4f} {val_acc_l1:>10.4f} "
      f"{train_acc_l1 - val_acc_l1:>12.4f} "
      f"{val_loss_l1  - train_loss_l1:>12.4f}")
print(f"{'L2 Regularization':<22} {train_acc_l2:>10.4f} {val_acc_l2:>10.4f} "
      f"{train_acc_l2 - val_acc_l2:>12.4f} "
      f"{val_loss_l2  - train_loss_l2:>12.4f}")
print(f"{'L1+L2 (Elastic Net)':<22} {train_acc_l1_l2:>10.4f} {val_acc_l1_l2:>10.4f} "
      f"{train_acc_l1_l2 - val_acc_l1_l2:>12.4f} "
      f"{val_loss_l1_l2  - train_loss_l1_l2:>12.4f}")
print("=" * 72)
print("Gap (Acc)  = Train Acc  - Val Acc   (higher = more overfit)")
print("Gap (Loss) = Val Loss   - Train Loss (higher = more overfit)")

## 8 — Accuracy Curves: All Four Models

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
fig.suptitle('Train vs Validation Accuracy — All Models',
             fontsize=16, fontweight='bold')

configs = [
    (axes[0, 0], history_overfit, 'No Regularization'),
    (axes[0, 1], history_l1,      'L1 Regularization'),
    (axes[1, 0], history_l2,      'L2 Regularization'),
    (axes[1, 1], history_l1_l2,   'L1 + L2 (Elastic Net)'),
]

for ax, hist, title in configs:
    ax.plot(hist.history['accuracy'],     label='Train Acc', linewidth=2)
    ax.plot(hist.history['val_accuracy'], label='Val Acc',   linewidth=2)
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Accuracy')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('regularization_accuracy_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: regularization_accuracy_comparison.png")

## 9 — Loss Curves: All Four Models

In [ ]:
fig2, axes2 = plt.subplots(2, 2, figsize=(15, 10))
fig2.suptitle('Train vs Validation Loss — All Models',
              fontsize=16, fontweight='bold')

configs2 = [
    (axes2[0, 0], history_overfit, 'No Regularization — Loss'),
    (axes2[0, 1], history_l1,      'L1 Regularization — Loss'),
    (axes2[1, 0], history_l2,      'L2 Regularization — Loss'),
    (axes2[1, 1], history_l1_l2,   'L1 + L2 (Elastic Net) — Loss'),
]

for ax, hist, title in configs2:
    ax.plot(hist.history['loss'],     label='Train Loss', linewidth=2)
    ax.plot(hist.history['val_loss'], label='Val Loss',   linewidth=2)
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('regularization_loss_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: regularization_loss_comparison.png")

## 10 — Overlay: Validation Accuracy of All Models on One Plot

This chart makes it easy to compare which model generalizes best.

In [ ]:
fig3, ax3 = plt.subplots(figsize=(12, 6))

ax3.plot(history_overfit.history['val_accuracy'], label='No Regularization',
         linewidth=2.5, linestyle='-')
ax3.plot(history_l1.history['val_accuracy'],      label='L1 Regularization',
         linewidth=2.5, linestyle='--')
ax3.plot(history_l2.history['val_accuracy'],      label='L2 Regularization',
         linewidth=2.5, linestyle='-.')
ax3.plot(history_l1_l2.history['val_accuracy'],   label='L1 + L2 (Elastic Net)',
         linewidth=2.5, linestyle=':')

ax3.set_title('Validation Accuracy — All Models Overlaid',
              fontsize=14, fontweight='bold')
ax3.set_xlabel('Epoch', fontsize=12)
ax3.set_ylabel('Validation Accuracy', fontsize=12)
ax3.legend(fontsize=11)
ax3.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('val_accuracy_overlay.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: val_accuracy_overlay.png")

## 11 — Weight Distribution After Training

- **L1** pushes many weights to exactly 0 → tall narrow spike at 0
- **L2** shrinks all weights → distribution tighter around 0
- **Baseline** → wider spread, larger weights allowed

In [ ]:
def get_weights(model):
    """Flatten all kernel (weight) matrices from Dense layers."""
    all_w = []
    for layer in model.layers:
        if isinstance(layer, layers.Dense):
            all_w.extend(layer.get_weights()[0].flatten())
    return np.array(all_w)

w_base  = get_weights(model_overfit)
w_l1    = get_weights(model_l1)
w_l2    = get_weights(model_l2)
w_l1l2  = get_weights(model_l1_l2)

fig4, ax4 = plt.subplots(figsize=(12, 5))

ax4.hist(w_base,  bins=50, alpha=0.55, label='No Regularization', density=True)
ax4.hist(w_l1,    bins=50, alpha=0.55, label='L1 Regularization',  density=True)
ax4.hist(w_l2,    bins=50, alpha=0.55, label='L2 Regularization',  density=True)
ax4.hist(w_l1l2,  bins=50, alpha=0.55, label='L1 + L2',            density=True)

ax4.set_title('Weight Distribution After Training', fontsize=14, fontweight='bold')
ax4.set_xlabel('Weight Value', fontsize=12)
ax4.set_ylabel('Density', fontsize=12)
ax4.legend(fontsize=11)
ax4.grid(True, alpha=0.3)
ax4.set_xlim([-1.5, 1.5])

print(f"Weight stats (std dev):")
print(f"  No Reg : {w_base.std():.4f}")
print(f"  L1     : {w_l1.std():.4f}")
print(f"  L2     : {w_l2.std():.4f}")
print(f"  L1+L2  : {w_l1l2.std():.4f}")
print(f"\nZero weights (|w| < 0.001):")
print(f"  No Reg : {(np.abs(w_base)  < 0.001).sum()} / {len(w_base)}")
print(f"  L1     : {(np.abs(w_l1)   < 0.001).sum()} / {len(w_l1)}")
print(f"  L2     : {(np.abs(w_l2)   < 0.001).sum()} / {len(w_l2)}")
print(f"  L1+L2  : {(np.abs(w_l1l2) < 0.001).sum()} / {len(w_l1l2)}")

plt.tight_layout()
plt.savefig('weight_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: weight_distribution.png")

## 12 — Key Takeaways

| Regularization | Penalty | Gradient | Effect |
|---|---|---|---|
| **None** | — | — | Weights grow freely → overfit |
| **L1** | `λ·Σ\|w\|` | `λ·sign(w)` (constant) | Sparsity — some weights → exactly 0 |
| **L2** | `λ·Σw²` | `2λw` (proportional) | Shrinkage — all weights stay small |
| **L1+L2** | both | both | Sparse + small weights |

### When to use what
- **L2** → default choice, works well in most neural networks
- **L1** → when you suspect many features/weights are irrelevant
- **L1+L2** → when you want feature selection AND smooth weight shrinkage
- **Start with** `λ = 0.001`, tune up if still overfitting